In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn

from tqdm import tqdm
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds
import tensorflow as tf

In [2]:
df_train = pd.read_csv("train_processed.csv")
df_test = pd.read_csv("test_processed.csv")

In [3]:
MAX_CHARS = 10000
SEED = 42
NUM_CLASSES = 2
BATCH_SIZE = 16
EPOCHS = 100
EMBEDDING_DIM = 100
MAX_LENGTH = 5000
DROPOUT = 0.5
LR = 0.0005

In [4]:
# Convert columns to string and integer types to prevent errors
df_train['file_1'] = df_train['file_1'].astype(str)
df_train['file_2'] = df_train['file_2'].astype(str)
df_train['real_text_id'] = df_train['real_text_id'].astype(int)

texts = []
labels = []

for _, row in df_train.iterrows():
    if row['real_text_id'] == 1:
        texts.append(row['file_1'][:MAX_CHARS])
        labels.append(1)
        texts.append(row['file_2'][:MAX_CHARS])
        labels.append(0)
    else:
        texts.append(row['file_1'][:MAX_CHARS])
        labels.append(0)
        texts.append(row['file_2'][:MAX_CHARS])
        labels.append(1)

data = pd.DataFrame({'text': texts,
                     'label': labels})

In [5]:
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    data, target_vocab_size=2**16)

In [6]:
def function_tokenize(text):
    return tokenizer.encode(text)

In [7]:
input_ids = data["text"].apply(function_tokenize)

input_ids = tf.keras.preprocessing.sequence.pad_sequences(input_ids,
                                                            value=0,
                                                            padding='post',
                                                            maxlen= MAX_LENGTH)

data["input_ids"] = input_ids.tolist()

data.drop(columns=["text"], inplace=True)

train_data = data.sample(frac=0.8, random_state=SEED).reset_index(drop=True)
test_data = data.drop(train_data.index).reset_index(drop=True)

In [8]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts = df["input_ids"]
        self.labels = df["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [9]:
train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)

train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=BATCH_SIZE,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=BATCH_SIZE)

In [10]:
class ModelClassificator(nn.Module):
    def __init__(self, num_classes, embedding, embedding_dim, dropout):
        super().__init__()

        self.embedding = embedding
        self.dropout = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(512)
        self.norm3 = nn.LayerNorm(256)
        self.norm4 = nn.LayerNorm(128)

        self.lstm1 = nn.LSTM(embedding_dim, 256, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(512, 128, batch_first=True, bidirectional=True)
        self.lstm3 = nn.LSTM(256, 64, batch_first=True, bidirectional=True)

        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
      x = self.embedding(x)
      x = self.norm1(x)
      # print("Shape", x.shape)
      # print("Output:\n", x.detach().numpy())

      x, _ = self.lstm1(x)
      x = self.dropout(x)
      x = self.norm2(x)
      # print("\nShape:", x.shape)
      # print("Output:\n", x.detach().numpy())

      x, _ = self.lstm2(x)
      x = self.dropout(x)
      x = self.norm3(x)
      # print("\nShape:", x.shape)
      # print("Output:\n", x.detach().numpy())

      x, _ = self.lstm3(x)
      x = self.dropout(x)
      x = self.norm4(x)
      # print("\nShape:", x.shape)
      # print("Output:\n", x.detach().numpy())

      x, _ = torch.max(x, dim=1)
      # print("\nShape:", x.shape)
      # print("Output:\n", x.detach().numpy())

      return self.classifier(x)

In [11]:
# input = torch.tensor(train_data['input_ids'][0], dtype=torch.long).unsqueeze(0)
# VOCAB_SIZE = tokenizer.vocab_size
# embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=0)
# model = ModelClassificator(NUM_CLASSES, embedding, EMBEDDING_DIM, DROPOUT)
# output = model(input)
# print('\nOutput:', output.detach().numpy())

In [12]:
class Trainer:
    def __init__(self,
             model, num_epochs, optimizer, criterion,
             train_data, test_data, scheduler, DEBUG=True):

        self.model = model
        self.num_epochs = num_epochs
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler
        self.train_data = train_data
        self.test_data = test_data
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.train_losses, self.test_losses = [], []
        self.train_accuracies, self.test_accuracies = [], []
        self.debug = DEBUG


    def train_epoch(self, epoch):
        """
        Executa o treinamento do modelo por uma época.

        Args:
            epoch: Número da época atual

        Returns:
            tuple: Média da perda e acurácia de treinamento
        """
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(self.train_data, desc=f'Epoch {epoch+1}/{self.num_epochs} [Train]')
        for data, target in pbar:
            data, target = data.to(self.device), target.to(self.device)
            self.optimizer.zero_grad() # Zera os gradientes acumulados
            output = self.model(data) # Passa os dados pelo modelo para obter as previsões
            loss = self.criterion(output, target) # Calcula a perda entre as previsões e os valores reais
            loss.backward()  # Calcula os gradientes da perda em relação aos parâmetros
            self.optimizer.step()  # Atualiza os pesos com base nos gradientes calculados

            running_loss += loss.item() # Acumula a perda para calcular a média
            _, predicted = output.max(1) # Encontra a classe com a maior probabilidade
            total += target.size(0) # Atualiza o total de amostras
            correct += predicted.eq(target).sum().item() # Conta o número de previsões corretas
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{100.*correct/total:.2f}%'}) # Atualiza o progresso

        return running_loss / len(self.train_data), 100. * correct / total # Retorna a média da perda e a acurácia

    def test_epoch(self):
        """
        Executa a avaliação do modelo.

        Returns:
            tuple: Média da perda e acurácia de teste
        """
        self.model.eval()  # Coloca o modelo em modo de avaliação (sem dropout, BN fixo)
        test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            pbar = tqdm(self.test_data, desc='Testing')
            for data, target in pbar:
                data, target = data.to(self.device), target.to(self.device) # Move os dados para o dispositivo
                output = self.model(data) # Passa os dados pelo modelo para obter as previsões
                test_loss += self.criterion(output, target).item() # Calcula a perda entre as previsões e os valores reais
                _, predicted = output.max(1) # Encontra a classe com a maior probabilidade
                total += target.size(0)
                correct += predicted.eq(target).sum().item()
                pbar.set_postfix({'Loss': f'{test_loss/len(self.test_data):.4f}', 'Acc': f'{100.*correct/total:.2f}%'})

        return test_loss / len(self.test_data), 100. * correct / total

    def train_test(self):
        """
        Executa o treinamento e avaliação do modelo.

        Returns:
            dict: Dicionário contendo métricas do treinamento e avaliação
        """
        self.model.to(self.device) # Move o modelo para o dispositivo

        model_name = self.model.__class__.__name__ # Obtém o nome do modelo
        print(f"Iniciando treinamento do modelo {model_name}")

        best_accuracy = 0.0

        for epoch in range(self.num_epochs):
            train_loss, train_acc = self.train_epoch(epoch) # Treina o modelo por uma época
            test_loss, test_acc = self.test_epoch() # Avalia o modelo no conjunto de teste

            self.scheduler.step() # Atualiza a taxa de aprendizado

            self.train_losses.append(train_loss) # Armazena a perda de treinamento
            self.train_accuracies.append(train_acc) # Armazena a acurácia de treinamento
            self.test_losses.append(test_loss) # Armazena a perda de teste
            self.test_accuracies.append(test_acc) # Armazena a acurácia de teste

            if test_acc > best_accuracy:
                best_accuracy = test_acc

            print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
                  f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}% | "
                  f"LR: {self.optimizer.param_groups[0]['lr']:.6f}")

        if self.debug:
            print(f"\nMelhor acurácia de teste: {best_accuracy:.2f}%")
        else:
          print(f"\nMelhor acurácia de teste: {best_accuracy:.2f}%")
          torch.save(self.model.state_dict(), f'{model_name.lower()}.pth')
          print(f"\nModelo salvo como {model_name.lower()}.pth")

        self.metrics = {
            'best_accuracy': best_accuracy,
            'train_losses': self.train_losses,
            'test_losses': self.test_losses,
            'train_accuracies': self.train_accuracies,
            'test_accuracies': self.test_accuracies
        }

    def __call__(self):
       return self.train_test()

    def plot_results(self):
        _, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6)) # Cria uma figura com dois subplots
        ax1.plot(self.train_losses, label='Train Loss') # Plota a perda de treinamento
        ax1.plot(self.test_losses, label='Test Loss') # Plota a perda de teste
        ax1.set_title('Histórico de Loss') # Define o título do primeiro subplot
        ax1.set_xlabel('Época') # Define o rótulo do eixo x do primeiro subplot
        ax1.set_ylabel('Loss') # Define o rótulo do eixo y do primeiro subplot
        ax1.legend() # Adiciona uma legenda ao primeiro subplot
        ax1.grid(True) # Adiciona uma grade ao primeiro subplot

        ax2.plot(self.train_accuracies, label='Train Accuracy') # Plota a acurácia de treinamento
        ax2.plot(self.test_accuracies, label='Test Accuracy') # Plota a acurácia de teste
        ax2.set_title('Histórico de Acurácia') # Define o título do segundo subplot
        ax2.set_xlabel('Época') # Define o rótulo do eixo x do segundo subplot
        ax2.set_ylabel('Acurácia (%)') # Define o rótulo do eixo y do segundo subplot
        ax2.legend() # Adiciona uma legenda ao segundo subplot
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig('training_results_improved.png', dpi=300)
        plt.show()

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

VOCAB_SIZE = tokenizer.vocab_size
embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=0)

base_model = ModelClassificator(NUM_CLASSES, embedding, EMBEDDING_DIM, DROPOUT)

optimizer = torch.optim.AdamW(base_model.parameters(),
                              lr=LR)

criterion=nn.CrossEntropyLoss()
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                    T_max=EPOCHS)

trainer = Trainer(
    base_model,
    EPOCHS,
    optimizer,
    criterion,
    train_loader,
    test_loader,
    scheduler,
    DEBUG=True
)

trainer()

In [ ]:
trainer.plot_results()

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
path_best_model = "/content/modelclassificator.pth"

model = ModelClassificator(NUM_CLASSES, embedding, EMBEDDING_DIM, dropout=DROPOUT)
model.load_state_dict(torch.load(path_best_model))
model.to(device)
model.eval()

ModelClassificator(
  (embedding): Embedding(259, 100, padding_idx=0)
  (dropout): Dropout(p=0.5, inplace=False)
  (norm1): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (norm4): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lstm1): LSTM(100, 256, batch_first=True, bidirectional=True)
  (lstm2): LSTM(512, 128, batch_first=True, bidirectional=True)
  (lstm3): LSTM(256, 64, batch_first=True, bidirectional=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)

In [24]:
def inference_pipeline(text):
    inputs = function_tokenize(text)

    with torch.no_grad():
        inputs = torch.tensor(inputs, dtype=torch.long).to(device)
        inputs = inputs.unsqueeze(0) # Add a batch dimension
        probs = model(inputs)
        probs = probs.detach().to('cpu')
        probs = F.softmax(probs, dim=1)
        return probs

def get_real_idx(prob_fake_file1, prob_fake_file2):
    if prob_fake_file1 < prob_fake_file2:
        return 1
    else:
        return 2

In [25]:
df_test.fillna("Article not found", inplace=True)

df_test["logits1"] = df_test.apply(lambda x: float(inference_pipeline(x["file_1"][:MAX_CHARS])[0][0]), axis=1)
df_test["logits2"] = df_test.apply(lambda x: float(inference_pipeline(x["file_2"][:MAX_CHARS])[0][0]), axis=1)
df_test['real_text_id'] = df_test.apply(lambda x: get_real_idx(x['logits1'], x['logits2']), axis=1)

df_sub = df_test[['id', 'real_text_id']]
df_sub.to_csv("submission.csv", index=False)
print("CSV Gerado")

CSV Gerado


In [28]:
values, counts = np.unique(df_sub['real_text_id'], return_counts=True)
print(values, counts)

[1 2] [567 501]
